In [ ]:
import scvi

# ============================================================
# SETUP: Tell scVI what to preserve vs what to integrate
# ============================================================

scvi.model.SCVI.setup_anndata(
    adata,
    batch_key='donor_id',           # Integrate across donors
    labels_key='disease_status',     # ← KEY: PRESERVE disease differences # # But: It's a balancing act, not a hard constraint

    continuous_covariate_keys=['age', 'sex']  # Other covariates
)

# Train model
vae = scvi.model.SCVI(
    adata,
    n_latent=30,
    gene_likelihood='nb'
)
vae.train(max_epochs=100, use_gpu=True)

# Get latent representation
adata.obsm['X_scvi'] = vae.get_latent_representation()

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

# Get latent dimensions
latents = adata.obsm['X_scvi']  # Shape: (n_cells, 30)

# ============================================================
# Identify which latents capture unwanted vs wanted variation
# ============================================================

latent_correlations = []

for i in range(latents.shape[1]):
    latent_i = latents[:, i]
    
    # Correlate with disease status (you want to KEEP this)
    disease_numeric = (adata.obs['disease_status'] == 'Diseased').astype(int)
    corr_disease, _ = pearsonr(latent_i, disease_numeric)
    
    # Correlate with technical variables (you want to REMOVE this)
    corr_lib_size, _ = pearsonr(latent_i, adata.obs['total_counts'])
    corr_mito, _ = pearsonr(latent_i, adata.obs['pct_counts_mt'])
    
    # Correlate with donor (nested with disease, but might have technical component)
    # Use per-donor mean to capture donor-level effects
    donor_means = adata.obs.groupby('donor_id')[f'latent_{i}'].transform('mean')
    corr_donor_mean, _ = pearsonr(latent_i, donor_means)
    
    latent_correlations.append({
        'latent': i,
        'disease_corr': abs(corr_disease),
        'lib_size_corr': abs(corr_lib_size),
        'mito_corr': abs(corr_mito),
        'donor_corr': abs(corr_donor_mean)
    })

corr_df = pd.DataFrame(latent_correlations)

# ============================================================
# SELECT latents that are:
# - Highly correlated with technical factors (lib_size, mito, donor)
# - Weakly correlated with disease
# ============================================================

technical_latents = corr_df[
    (corr_df['lib_size_corr'] > 0.3) |  # Strong technical correlation
    (corr_df['mito_corr'] > 0.3) |
    (corr_df['donor_corr'] > 0.3) &
    (corr_df['disease_corr'] < 0.2)      # Weak disease correlation
]['latent'].values

print(f"Technical latents: {technical_latents}")

# Add these as covariates
for i in technical_latents:
    adata.obs[f'scVI_tech_SV{i}'] = latents[:, i]